# 02 — Speaker-independent split and MFCC preprocessing
Run after completing the manifest from Notebook 01. This keeps all files from a speaker in one split and fits normalization on training data only.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, librosa, joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

OUT=Path("anu_revision_output")
df=pd.read_csv(OUT/"dataset_manifest_TO_FILL.csv").fillna("")
required=["path","label","speaker_id","original_file","is_augmented"]
for col in required:
    assert col in df.columns and (df[col].astype(str).str.strip()!="").all(), f"Complete column: {col}"
assert df.path.map(lambda p: Path(p).exists()).all(), "Some WAV paths do not exist."
assert (df.groupby("original_file").speaker_id.nunique()==1).all(), "An original and its derivatives have inconsistent speaker IDs."
df["is_augmented"]=df.is_augmented.astype(str).str.lower().isin(["true","1","yes"])

In [2]:
split=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=42)
itr,ite=next(split.split(df,groups=df.speaker_id))
tr,te=df.iloc[itr].copy(),df.iloc[ite].copy()
assert set(tr.speaker_id).isdisjoint(set(te.speaker_id))
print("Train speakers:",tr.speaker_id.nunique(),"Test speakers:",te.speaker_id.nunique())
print("Train files:",len(tr),"Test files:",len(te))
display(te.groupby("label").size().to_frame("test_count"))

Train speakers: 42 Test speakers: 11
Train files: 3522 Test files: 918


,test_count
label,
anu,66
close browser,66
close calculator,66
help,66
mute,60
next,66
open browser,66
open calculator,66
pause,66


In [3]:
SR,N_MFCC,FRAMES=22050,20,40
def mfcc(path):
    y,_=librosa.load(path,sr=SR,mono=True)
    x=librosa.feature.mfcc(y=y,sr=SR,n_mfcc=N_MFCC)
    if x.shape[1]<FRAMES: x=np.pad(x,((0,0),(0,FRAMES-x.shape[1])))
    else: x=x[:,:FRAMES]
    return x.astype("float32")
Xtr_raw=np.stack([mfcc(p) for p in tr.path])
Xte_raw=np.stack([mfcc(p) for p in te.path])
mean,std=float(Xtr_raw.mean()),float(Xtr_raw.std()+1e-6)
Xtr=((Xtr_raw-mean)/std)[...,None]
Xte=((Xte_raw-mean)/std)[...,None]
enc=LabelEncoder()
ytr=enc.fit_transform(tr.label)
yte=enc.transform(te.label)
np.savez_compressed(OUT/"speaker_split_mfcc.npz",X_train=Xtr,X_test=Xte,y_train=ytr,y_test=yte,
                    train_paths=tr.path.to_numpy(),test_paths=te.path.to_numpy(),
                    mean=mean,std=std,classes=enc.classes_)
joblib.dump(enc,OUT/"label_encoder.joblib")
tr.to_csv(OUT/"train_manifest.csv",index=False)
te.to_csv(OUT/"test_manifest.csv",index=False)
print("Saved MFCC arrays:",Xtr.shape,Xte.shape,"Classes:",list(enc.classes_))

Saved MFCC arrays: (3522, 20, 40, 1) (918, 20, 40, 1) Classes: ['anu', 'close browser', 'close calculator', 'help', 'mute', 'next', 'open browser', 'open calculator', 'pause', 'play', 'previous', 'screenshot', 'sleep', 'unmute']


In [5]:
import pandas as pd
from pathlib import Path

manifest = pd.read_csv("anu_revision_output/dataset_manifest_TO_FILL.csv")

print("Total unique speaker IDs:")
print(manifest["speaker_id"].nunique())

print("\nSpeaker IDs:")
print(sorted(manifest["speaker_id"].dropna().unique()))

print("\nSpeaker ID distribution:")
print(manifest.groupby("speaker_id").size())

Total unique speaker IDs:
53

Speaker IDs:
['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21', 'S22', 'S23', 'S24', 'S25', 'S26', 'S27', 'S28', 'S29', 'S30', 'S31', 'S32', 'S33', 'S34', 'S35', 'S36', 'S37', 'S38', 'S39', 'S40', 'S41', 'S42', 'S43', 'S44', 'S45', 'S46', 'S47', 'S48', 'S49', 'S51', 'S52', 'S53', 'S54']

Speaker ID distribution:
speaker_id
S01    90
S02    84
S03    84
S04    84
S05    84
S06    84
S07    84
S08    84
S09    84
S10    84
S11    84
S12    84
S13    84
S14    84
S15    84
S16    84
S17    84
S18    84
S19    84
S20    84
S21    84
S22    84
S23    90
S24    84
S25    84
S26    84
S27    84
S28    84
S29    84
S30    84
S31    84
S32    78
S33    78
S34    78
S35    78
S36    78
S37    84
S38    84
S39    84
S40    84
S41    84
S42    84
S43    84
S44    84
S45    84
S46    96
S47    84
S48    84
S49    84
S51    84
S52    84
S53    84
S54    78
dtype: int64


In [6]:
import pandas as pd

train = pd.read_csv("anu_revision_output/train_manifest.csv")
test = pd.read_csv("anu_revision_output/test_manifest.csv")

print("TRAIN")
print("Unique speaker IDs:", train["speaker_id"].nunique())
print("Total files:", len(train))
print("Original recordings:", train["original_file"].nunique())

print("\nTEST")
print("Unique speaker IDs:", test["speaker_id"].nunique())
print("Total files:", len(test))
print("Original recordings:", test["original_file"].nunique())

print("\nSpeaker IDs appearing in BOTH:")
print(set(train["speaker_id"]) & set(test["speaker_id"]))

TRAIN
Unique speaker IDs: 42
Total files: 3522
Original recordings: 587

TEST
Unique speaker IDs: 11
Total files: 918
Original recordings: 153

Speaker IDs appearing in BOTH:
set()
